# 02 Analisis & Pemodelan Machine Learning (Random Forest)
Notebook ini digunakan oleh **Iqbal & Adam** untuk:
1. Melakukan Eksplorasi Data Awal (EDA) terhadap data panel bersih.
2. Melatih model **Random Forest Regressor** dalam memprediksi **Persentase Kemiskinan** berbasis indikator **Rata-rata Lama Sekolah (RLS)** dan **Tingkat Pengangguran Terbuka (TPT)**.
3. Mengevaluasi performa model dan menyimpan berkas model ke `model/model_kemiskinan.pkl`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle

# Pindah ke root direktori proyek jika dijalankan dari dalam folder notebooks
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
print("Direktori kerja saat ini:", os.getcwd())

## 1. Memuat Dataset
Membaca data panel bersih yang merupakan hasil keluaran dari pipeline ETL (`data/data_bersih.csv`).

In [ ]:
clean_data_path = "data/data_bersih.csv"
if os.path.exists(clean_data_path):
    df = pd.read_csv(clean_data_path)
    print("Data bersih berhasil dimuat!")
    print(df.head())
else:
    print(f"[!] Berkas {clean_data_path} tidak ditemukan.")
    print("    Silakan jalankan notebook '01_pipeline_etl.ipynb' terlebih dahulu untuk menghasilkan data bersih.")

## 2. Exploratory Data Analysis (EDA)
Melakukan analisis deskriptif dasar dan visualisasi korelasi antar variabel.

In [ ]:
if 'df' in locals():
    print("--- Statistik Deskriptif ---")
    print(df.describe())
    
    # Matriks Korelasi
    plt.figure(figsize=(8, 6))
    sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Matriks Korelasi Indikator")
    plt.show()

## 3. Persiapan Data & Pembagian Dataset
Menentukan Fitur (X) dan Target (y), serta membagi data menjadi set pelatihan (train) dan pengujian (test).

In [ ]:
if 'df' in locals():
    # Membersihkan baris yang bernilai null pada fitur/target
    df_clean = df.dropna(subset=["Rata_rata_Lama_Sekolah", "Tingkat_Pengangguran_Terbuka", "Persentase_Kemiskinan"])
    
    # Menetapkan X dan y
    X = df_clean[["Rata_rata_Lama_Sekolah", "Tingkat_Pengangguran_Terbuka"]]
    y = df_clean["Persentase_Kemiskinan"]
    
    # Split data 80% train, 20% test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Ukuran Data Train: {X_train.shape[0]} baris")
    print(f"Ukuran Data Test : {X_test.shape[0]} baris")

## 4. Pelatihan Model & Evaluasi
Melatih model regresi berbasis ensemble Random Forest dan mengukur akurasinya menggunakan metrik regresi standar.

In [ ]:
if 'df' in locals():
    # Inisialisasi RandomForestRegressor
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    
    # Fitting Model
    rf_model.fit(X_train, y_train)
    
    # Prediksi data pengujian
    y_pred = rf_model.predict(X_test)
    
    # Metrik Evaluasi
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print("--- Hasil Evaluasi Random Forest Regressor ---")
    print(f"Mean Squared Error (MSE)      : {mse:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Mean Absolute Error (MAE)     : {mae:.4f}")
    print(f"R-squared (R2 Score)          : {r2:.4f}")

## 5. Menyimpan Model Terlatih
Mengekspor model terlatih ke folder `model/model_kemiskinan.pkl` dalam format pickle agar dapat dimuat kembali nanti (misalnya untuk deployment).

In [ ]:
if 'rf_model' in locals():
    model_dir = "model"
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "model_kemiskinan.pkl")
    
    with open(model_path, "wb") as f:
        pickle.dump(rf_model, f)
        
    print(f"[+] Model sukses disimpan ke: {model_path}")